In [5]:
# STEP 1: Importing Libraries

import pandas as pd
import numpy as np

df = pd.read_csv('marketing_campaign_data_messy.csv')

print(f"Loaded Datasets: {df.shape[0]} rows, {df.shape[1]} columns")

Loaded Datasets: 2020 rows, 12 columns


In [6]:
df

,Campaign_ID,Campaign_Name,Start_Date,End_Date,Channel,Impressions,Clicks,Spend,Conversions,Active,Clicks,Campaign_Tag
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24 00:00:00,2023-12-13,TikTok,16795,197,$102.82,20.0,Y,NaN,TI
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06 00:00:00,2023-05-12,Facebook,1860,30,24.33,1.0,0,NaN,FA
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13 00:00:00,2023-12-20,Email,77820,843,1323.39,51.0,No,NaN,EM
3,CMP-00004,Q1_BlackFriday_CMP-00004,2023-10-30,2023-11-03,TikTok,55886,2019,2180.38,135.0,True,NaN,TI
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22 00:00:00,2023-04-23,Facebook,7265,169,252.44,30.0,Yes,NaN,FA
...,...,...,...,...,...,...,...,...,...,...,...,...
2015,CMP-00400,Q3_Summer_CMP-00400,2023-10-31 00:00:00,2023-11-13,TikTok,30592,586,$503.95,77.0,1,NaN,TI
2016,CMP-01255,Q4_Summer_CMP-01255,2023-09-01 00:00:00,2023-09-26,Google Ads,20097,897,1641.0,162.0,0,NaN,GO
2017,CMP-01050,Q2_Launch_CMP-01050,2023-02-09 00:00:00,2023-02-21,Instagram,33254,1117,883.82,214.0,0,NaN,IN
2018,CMP-01118,Q4_Winter_CMP-01118,2023-03-30 00:00:00,2023-04-27,Facebook,68728,2960,4198.5,591.0,Yes,NaN,FA


In [7]:
# STEP 2: Column Name Cleaning

print(df.columns.tolist())

df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

print(df.columns.tolist())

[' Campaign_ID ', 'Campaign_Name', 'Start_Date', 'End_Date', 'Channel', 'Impressions', 'Clicks ', 'Spend', 'Conversions', 'Active', 'Clicks', 'Campaign_Tag']
['campaign_id', 'campaign_name', 'start_date', 'end_date', 'channel', 'impressions', 'clicks', 'spend', 'conversions', 'active', 'clicks', 'campaign_tag']


In [8]:
# STEP 3: Data Type Conversion && Symbol Cleaning

dirty_spend_mask = df['spend'].astype(str).str.contains(r'\$')
print(df.loc[dirty_spend_mask, ['campaign_id', 'spend']].head(3))

df['spend'] = df['spend'].astype(str).str.replace(r'[^\d.-]', '', regex=True)
df['spend'] = pd.to_numeric(df['spend'])

print("Fix applied perfectly")
print(df.loc[dirty_spend_mask,['campaign_id', 'spend']].head(3))

   campaign_id     spend
0    CMP-00001   $102.82
21   CMP-00022   $2428.4
22   CMP-00023  $4726.22
Fix applied perfectly
   campaign_id    spend
0    CMP-00001   102.82
21   CMP-00022  2428.40
22   CMP-00023  4726.22


In [9]:
# Step 4: Channel Typographical Errors Correction (Fuzzy Logic)

print(df['channel'].unique())
cleanup_nap = {
    'Facebok': 'Facebook',
    'Tik_Tok': 'TikTok',
    'E-mail': 'Email',
    'Insta_gram': 'Instagram',
    'Gogle': 'Google Ads',
    'N/A': np.nan #Handling the ghost value
}

df['channel'] = df['channel'].replace(cleanup_nap)

print("Fix applied")
print(df['channel'].unique())

['TikTok' 'Facebook' 'Email' 'Instagram' 'Google Ads' 'E-mail' nan 'Gogle'
 'Tik_Tok' 'Facebok' 'Insta_gram']
Fix applied
['TikTok' 'Facebook' 'Email' 'Instagram' 'Google Ads' nan]


In [19]:
# Step 5: Active Boolean Conversion (Fuzzy Logic)

print(df['active'].unique())

bool_map = {
    'Yes': True,
    'Y': True,
    1: True,
    'No': False,
    0: False,
}

df['active'] = df['active'].map(bool_map).fillna(False).astype(bool)

print("Fix applied")
print(df['active'].unique())

[ True False]
Fix applied
[ True False]


In [11]:
# Step 6: Date Parsing

print(df['start_date'].dtype)

df['start_date'] = pd.to_datetime(df['start_date'], errors='coerce')
df['end_date'] = pd.to_datetime(df['end_date'], dayfirst=True, errors='coerce')

print("Fix applied")
print(df['start_date'].dtype)

object
Fix applied
datetime64[ns]


/tmp/ipykernel_2047/1736553292.py:6: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['end_date'] = pd.to_datetime(df['end_date'], dayfirst=True, errors='coerce')


In [12]:
# Step 7: Column Duplication Cleaning
df = df.loc[:, ~df.columns.duplicated()]
df

,campaign_id,campaign_name,start_date,end_date,channel,impressions,clicks,spend,conversions,active,campaign_tag
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24,2023-12-13,TikTok,16795,197,102.82,20.0,True,TI
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06,2023-05-12,Facebook,1860,30,24.33,1.0,False,FA
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13,2023-12-20,Email,77820,843,1323.39,51.0,False,EM
3,CMP-00004,Q1_BlackFriday_CMP-00004,NaT,2023-11-03,TikTok,55886,2019,2180.38,135.0,False,TI
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22,2023-04-23,Facebook,7265,169,252.44,30.0,True,FA
...,...,...,...,...,...,...,...,...,...,...,...
2015,CMP-00400,Q3_Summer_CMP-00400,2023-10-31,2023-11-13,TikTok,30592,586,503.95,77.0,False,TI
2016,CMP-01255,Q4_Summer_CMP-01255,2023-09-01,2023-09-26,Google Ads,20097,897,1641.00,162.0,False,GO
2017,CMP-01050,Q2_Launch_CMP-01050,2023-02-09,2023-02-21,Instagram,33254,1117,883.82,214.0,False,IN
2018,CMP-01118,Q4_Winter_CMP-01118,2023-03-30,2023-04-27,Facebook,68728,2960,4198.50,591.0,True,FA


In [13]:
# Step 8: Logic Integrity (Clicks vs. Impressions)

impossible_mask = df['clicks'] > df['impressions']
print(df.loc[impossible_mask, ['campaign_id', 'impressions', 'clicks']].head(3))

Empty DataFrame
Columns: [campaign_id, impressions, clicks]
Index: []


In [14]:
# Step 9: Logical Integrity (End_date vs. Start_date)

time_travel_mask = df['end_date'] < df['start_date']
print(df.loc[time_travel_mask, ['campaign_id', 'start_date', 'end_date']].head(3))

df.loc[time_travel_mask, 'end_date'] = df.loc[time_travel_mask, 'start_date'] + pd.Timedelta(days=30)

print("Fix applied")
print(df.loc[time_travel_mask, ['campaign_id', 'start_date', 'end_date']].head(3))

   campaign_id start_date   end_date
23   CMP-00024 2023-05-06 2023-05-01
54   CMP-00055 2023-09-01 2023-08-27
71   CMP-00072 2023-02-01 2023-01-27
Fix applied
   campaign_id start_date   end_date
23   CMP-00024 2023-05-06 2023-06-05
54   CMP-00055 2023-09-01 2023-10-01
71   CMP-00072 2023-02-01 2023-03-03


In [15]:
# Step 10: Checking Ouliers

q1 = df['spend'].quantile(0.25)
q3 = df['spend'].quantile(0.75)

IQR = q3 - q1
upper_limit = q3 + (3 * IQR)

outlier_mask = df['spend'] > upper_limit
print(df.loc[outlier_mask, ['campaign_id', 'spend']].head(3))

print("Fix applied")
df.loc[outlier_mask, 'spend'] = upper_limit
print(df.loc[outlier_mask, ['campaign_id', 'spend']].head(3))

     campaign_id      spend
789    CMP-00790  500000.00
1443   CMP-01444    8921.51
1460   CMP-01461  500000.00
Fix applied
     campaign_id      spend
789    CMP-00790  8603.5375
1443   CMP-01444  8603.5375
1460   CMP-01461  8603.5375


In [20]:
# Step 11:  String Parsing

print(df['campaign_name'].head(3))

df['season'] = df['campaign_name'].str.extract(r'Q\d_([^_]+)_')

print("Fix applied")
print(df[['season', 'campaign_name']].head(3))

0    Q4_Summer_CMP-00001
1    Q1_Launch_CMP-00002
2    Q3_Winter_CMP-00003
Name: campaign_name, dtype: object
Fix applied
   season        campaign_name
0  Summer  Q4_Summer_CMP-00001
1  Launch  Q1_Launch_CMP-00002
2  Winter  Q3_Winter_CMP-00003


In [17]:
df

,campaign_id,campaign_name,start_date,end_date,channel,impressions,clicks,spend,conversions,active,campaign_tag,season
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24,2023-12-13,TikTok,16795,197,102.82,20.0,True,TI,Summer
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06,2023-05-12,Facebook,1860,30,24.33,1.0,False,FA,Launch
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13,2023-12-20,Email,77820,843,1323.39,51.0,False,EM,Winter
3,CMP-00004,Q1_BlackFriday_CMP-00004,NaT,2023-11-03,TikTok,55886,2019,2180.38,135.0,False,TI,BlackFriday
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22,2023-04-23,Facebook,7265,169,252.44,30.0,True,FA,Winter
...,...,...,...,...,...,...,...,...,...,...,...,...
2015,CMP-00400,Q3_Summer_CMP-00400,2023-10-31,2023-11-13,TikTok,30592,586,503.95,77.0,False,TI,Summer
2016,CMP-01255,Q4_Summer_CMP-01255,2023-09-01,2023-09-26,Google Ads,20097,897,1641.00,162.0,False,GO,Summer
2017,CMP-01050,Q2_Launch_CMP-01050,2023-02-09,2023-02-21,Instagram,33254,1117,883.82,214.0,False,IN,Launch
2018,CMP-01118,Q4_Winter_CMP-01118,2023-03-30,2023-04-27,Facebook,68728,2960,4198.50,591.0,True,FA,Winter


In [21]:
print("=== 1. DATAFRAME INFO ===")
df.info()

print("\n=== 2. UNIQUE CHANNELS (Cleaned) ===")
print(df['channel'].unique())

print("\n=== 3. ACTIVE VALUE COUNTS (Strictly Boolean) ===")
print(df['active'].value_counts())

print("\n=== 4. CHECK TIME TRAVEL LOGIC (Should be 0) ===")
invalid_dates = (df['end_date'] < df['start_date']).sum()
print(f"Rows where end_date < start_date: {invalid_dates}")

# Save the cleaned dataset to a new file
df.to_csv('marketing_campaign_data_cleaned.csv', index=False)
print("\n[SUCCESS] Cleaned dataset saved to 'marketing_campaign_data_cleaned.csv'")

=== 1. DATAFRAME INFO ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2020 entries, 0 to 2019
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   campaign_id    2020 non-null   object        
 1   campaign_name  2020 non-null   object        
 2   start_date     1687 non-null   datetime64[ns]
 3   end_date       2020 non-null   datetime64[ns]
 4   channel        1919 non-null   object        
 5   impressions    2020 non-null   int64         
 6   clicks         2020 non-null   int64         
 7   spend          2020 non-null   float64       
 8   conversions    1820 non-null   float64       
 9   active         2020 non-null   bool          
 10  campaign_tag   2020 non-null   object        
 11  season         2020 non-null   object        
dtypes: bool(1), datetime64[ns](2), float64(2), int64(2), object(5)
memory usage: 175.7+ KB

=== 2. UNIQUE CHANNELS (Cleaned) ===
['TikTok' 'Facebook' 'E

```markdown
### Programmatically Push Cleaned CSV to GitHub
To run this code, you will need:
1. A GitHub Personal Access Token (PAT) with `repo` scope permissions.
2. Your GitHub Username and Repository name.

Fill in your details in the cell below and run it to push the cleaned CSV file directly to your repository.
```